# Advanced NN: MNIST Classification with Regularization

In this exercise we explore **regularization techniques** for neural networks using the MNIST handwritten digit dataset.

We will build four variants of a Multi-Layer Perceptron (MLP) and compare their performance:
1. **Baseline** -- no regularization
2. **Dropout** -- randomly zero out activations during training
3. **Batch Normalization** -- normalize activations between layers
4. **L2 Regularization** (weight decay) -- penalize large weights

**Dataset:** MNIST (28x28 greyscale images of digits 0-9, downloaded once via `torchvision` into `./data`; set the `COURSE_DATA` environment variable to use a copy prepared before class).

**Evaluation protocol:** 10,000 of the 60,000 training images are held out as a **validation set**. It is used for per-epoch monitoring, for choosing each model's checkpoint (the epoch with the lowest validation loss), and for comparing the four models. The 10,000 test images are used **once**, at the end, for the model chosen on validation.

**Runtime:** 4 models × 10 epochs on 50,000 training images, about 3 minutes on a recent laptop CPU and up to about 10 minutes on an older one. If time is short, set `NUM_EPOCHS = 5` in section 3.

**Learning objectives:**
1. Understand and implement common regularization strategies.
2. Compare training dynamics and generalization across variants on validation data.
3. Visualize misclassified examples to gain intuition.

---
## 1. Imports

In [ ]:
import copy
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms

from sklearn.metrics import classification_report

SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}, device: {DEVICE}")

---
## 2. Load MNIST, Hold Out a Validation Set, and Visualize Samples

We split the 60,000 training images into 50,000 for training and 10,000 for validation with a seeded `random_split`. The validation images drive every decision in this notebook (checkpoint epoch, model comparison); the test images stay untouched until section 9. `download=True` only downloads when the files are missing, so the notebook also runs offline once `DATA_ROOT` holds MNIST.

The normalization constants (0.1307, 0.3081) are the published mean and standard deviation of the MNIST training images; the test images play no part in them (recomputed on our 50,000 training images they agree to three decimals).

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST training-set mean, std
])

DATA_ROOT = os.environ.get("COURSE_DATA", "./data")   # folder that holds (or receives) MNIST
full_train = torchvision.datasets.MNIST(root=DATA_ROOT, train=True,
                                        download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root=DATA_ROOT, train=False,
                                          download=True, transform=transform)

# Hold out 10,000 training images for validation (seeded, reproducible split)
train_dataset, val_dataset = random_split(
    full_train, [50_000, 10_000], generator=torch.Generator().manual_seed(SEED))

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1000, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)   # used once, section 9

print(f"Train samples: {len(train_dataset)}, validation samples: {len(val_dataset)}, "
      f"test samples: {len(test_dataset)}")

In [ ]:
# Visualize a grid of sample digits
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.suptitle("Sample MNIST Digits")
plt.tight_layout()
plt.show()

---
## 3. Helper Functions: Train and Evaluate

`train_model` trains for a fixed budget of `NUM_EPOCHS`, evaluates the **validation** set after every epoch, keeps a copy of the weights from the epoch with the lowest validation loss, and restores that checkpoint at the end. The test set is never used during training.

In [ ]:
NUM_EPOCHS = 10   # per model; set to 5 if time is short


def evaluate_loss_acc(model, loader, criterion, device=DEVICE):
    """Mean loss and accuracy on a loader, in eval mode (no dropout, BatchNorm running stats)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            X_batch = X_batch.view(X_batch.size(0), -1)
            logits = model(X_batch)
            total_loss += criterion(logits, y_batch).item() * X_batch.size(0)
            correct += (logits.argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)
    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, optimizer, criterion,
                num_epochs=NUM_EPOCHS, device=DEVICE):
    """Train model, monitor the validation set every epoch, restore the best-validation checkpoint."""
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    model.to(device)
    best_loss, best_state, best_epoch = float("inf"), None, 0

    for epoch in range(1, num_epochs + 1):
        # --- Train ---
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            X_batch = X_batch.view(X_batch.size(0), -1)  # flatten 28x28 -> 784
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X_batch.size(0)
            correct += (logits.argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)
        history["train_loss"].append(total_loss / total)   # running average, train mode
        history["train_acc"].append(correct / total)

        # --- Validate (never the test set) ---
        val_loss, val_acc = evaluate_loss_acc(model, val_loader, criterion, device)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch
            best_state = copy.deepcopy(model.state_dict())

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:2d}  "
                  f"Train Loss: {history['train_loss'][-1]:.4f}  "
                  f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

    model.load_state_dict(best_state)   # restore the best-validation checkpoint
    history["best_epoch"] = best_epoch
    print(f"  Restored epoch {best_epoch} (lowest validation loss {best_loss:.4f})")
    return history


def evaluate_model(model, loader, device=DEVICE):
    """Return (y_true, y_pred) arrays for a loader."""
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device).view(X_batch.size(0), -1)
            preds = model(X_batch).argmax(1).cpu()
            all_true.append(y_batch)
            all_pred.append(preds)
    return torch.cat(all_true).numpy(), torch.cat(all_pred).numpy()

---
## 4. Baseline MLP (No Regularization)

In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


print("=== Baseline MLP ===")
torch.manual_seed(SEED)
baseline_model = BaselineMLP().to(DEVICE)
baseline_opt = optim.Adam(baseline_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
baseline_hist = train_model(baseline_model, train_loader, val_loader, baseline_opt, criterion)

---
## 5. MLP with Dropout

In [ ]:
class DropoutMLP(nn.Module):
    def __init__(self, drop_rate=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


print("=== Dropout MLP ===")
torch.manual_seed(SEED)
dropout_model = DropoutMLP().to(DEVICE)
dropout_opt = optim.Adam(dropout_model.parameters(), lr=1e-3)
dropout_hist = train_model(dropout_model, train_loader, val_loader, dropout_opt, criterion)

---
## 6. MLP with Batch Normalization

In [ ]:
class BatchNormMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


print("=== BatchNorm MLP ===")
torch.manual_seed(SEED)
bn_model = BatchNormMLP().to(DEVICE)
bn_opt = optim.Adam(bn_model.parameters(), lr=1e-3)
bn_hist = train_model(bn_model, train_loader, val_loader, bn_opt, criterion)

---
## 7. MLP with L2 Regularization (Weight Decay)

Adam's `weight_decay` adds an L2 penalty to the gradient (the coupled form); `optim.AdamW` decouples the decay from the adaptive step. Comparing the two is an extension.

In [ ]:
print("=== L2 (Weight Decay) MLP ===")
torch.manual_seed(SEED)
l2_model = BaselineMLP().to(DEVICE)  # same architecture, different optimizer
l2_opt = optim.Adam(l2_model.parameters(), lr=1e-3, weight_decay=1e-4)
l2_hist = train_model(l2_model, train_loader, val_loader, l2_opt, criterion)

---
## 8. Comparison Table and Chart (validation set)

We compare the four models at their restored checkpoints on the **validation** set and choose the one with the highest validation accuracy. The training accuracy in the table is re-measured in eval mode on the 50,000 training images: the per-epoch training accuracy is a running average in train mode (dropout active, weights still changing), which would understate the gap for the dropout model.

The four models usually land close together. Before ranking them, repeat the notebook with a second seed (change `SEED`): a difference smaller than the seed-to-seed change is not evidence.

In [ ]:
models = {
    "Baseline": (baseline_model, baseline_hist),
    "Dropout": (dropout_model, dropout_hist),
    "BatchNorm": (bn_model, bn_hist),
    "L2 (weight_decay)": (l2_model, l2_hist),
}
train_eval_loader = DataLoader(train_dataset, batch_size=1000, shuffle=False)

rows = []
for name, (m, h) in models.items():
    _, train_acc = evaluate_loss_acc(m, train_eval_loader, criterion)
    val_loss, val_acc = evaluate_loss_acc(m, val_loader, criterion)
    rows.append({"Model": name, "Selected Epoch": h["best_epoch"], "Train Acc": train_acc,
                 "Val Acc": val_acc, "Val Loss": val_loss,
                 "Generalization Gap": train_acc - val_acc})
results = pd.DataFrame(rows).round(4)
print(results.to_string(index=False))

best_name = results.loc[results["Val Acc"].idxmax(), "Model"]
best_model = models[best_name][0]
print(f"\nChosen on validation accuracy: {best_name}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, (m, h) in models.items():
    epochs = range(1, len(h["val_loss"]) + 1)
    axes[0].plot(epochs, h["val_loss"], label=name)
    axes[1].plot(epochs, h["val_acc"], label=name)

axes[0].set_title("Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Final Test Evaluation and Misclassified Examples

The model chosen on validation is now evaluated **once** on the 10,000 test images: this is the only use of the test set. We then look at some of the test images it got wrong. Do not go back and change the model because of this result.

In [ ]:
# The only use of the test set: the model chosen on validation, evaluated once
y_true, y_pred = evaluate_model(best_model, test_loader)
test_acc = (y_true == y_pred).mean()
misclassified_idx = np.where(y_true != y_pred)[0]
print(f"{best_name}: test accuracy {test_acc:.4f}, misclassified {len(misclassified_idx)} / {len(y_true)}")

fig, axes = plt.subplots(2, 8, figsize=(14, 4.6))
for i, ax in enumerate(axes.flat):
    if i >= len(misclassified_idx):
        ax.axis("off")
        continue
    idx = misclassified_idx[i]
    img, true_label = test_dataset[idx]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"T:{true_label} P:{y_pred[idx]}", fontsize=9)
    ax.axis("off")
plt.suptitle(f"Misclassified Test Examples, {best_name} (T=True, P=Predicted)")
plt.tight_layout()
plt.show()

In [ ]:
print(f"Classification Report ({best_name}, test set):")
print(classification_report(y_true, y_pred, digits=4))

---
## 10. Summary

In this notebook we:

1. Loaded MNIST via `torchvision`, held out 10,000 training images for validation, and visualized sample digits.
2. Built a **baseline** MLP with no regularization.
3. Added **Dropout** -- randomly zeros activations to reduce co-adaptation.
4. Added **Batch Normalization** -- stabilizes and often accelerates training.
5. Added **L2 regularization** via `weight_decay` in the optimizer.
6. Compared all four variants on validation accuracy and generalization gap, each at its best-validation checkpoint.
7. Evaluated the chosen model once on the test set and visualized misclassified examples.

**Key takeaways:**
- Regularization helps close the gap between training and validation performance.
- BatchNorm often speeds up training; dropout and weight decay target the train-validation gap -- measure whether they narrow it here.
- On MNIST the variants differ by small margins: check a difference against a second seed before ranking.
- Validation data choose the checkpoint and the model; the test set is used once, for the final report.